In [ ]:
## CONFIGURATION — change these to analyze a different dataset / backbone / fold.
## Every other cell in this notebook derives its paths and labels from these values.
DATASET_FLAG = 'tissuemnist'
MODEL_BACKBONE = 'resnet18'
FOLD_IDX = 1
IMAGE_SIZE = 224
COLOR = DATASET_FLAG in ['dermamnist', 'dermamnist-e', 'pathmnist', 'bloodmnist', 'hmu-crc']

SHAP_CACHE_DIR = '/workspace/uq_benchmark_results/shap_cache'
RESULTS_CACHE_DIR = '/workspace/uq_benchmark_results/cache'

SHAP_CACHE_FILE = f'{SHAP_CACHE_DIR}/shap_cache_{DATASET_FLAG}_{MODEL_BACKBONE}_fold{FOLD_IDX}_bg1000.npz'
TEST_CACHE_FILE = f'{RESULTS_CACHE_DIR}/{DATASET_FLAG}_{MODEL_BACKBONE}_test_results.npz'
CALIB_CACHE_FILE = f'{RESULTS_CACHE_DIR}/{DATASET_FLAG}_{MODEL_BACKBONE}_calib_results.npz'

print(f"Config: dataset={DATASET_FLAG}, backbone={MODEL_BACKBONE}, fold={FOLD_IDX}, color={COLOR}")
print(f"  SHAP cache : {SHAP_CACHE_FILE}")
print(f"  Test cache : {TEST_CACHE_FILE}")
print(f"  Calib cache: {CALIB_CACHE_FILE}")


# KNN-SHAP Investigation

Investigation notebook to understand why KNN_SHAP is underperforming compared to KNN_Raw.

## Key Questions:
1. Are the selected SHAP features discriminative?
2. Do selected features differ significantly between classes?
3. Is the issue with calibration vs test set?
4. Are there issues with feature overlap or redundancy?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

## 1. Load SHAP Cache Data

In [ ]:
# Load SHAP cache file
cache_file = SHAP_CACHE_FILE
cache = np.load(cache_file, allow_pickle=True)

# Extract data
shap_values = cache['shap_values']  # (N_calib, n_features, n_classes)
features = cache['features']  # (N_calib, n_features)
labels = cache['labels']  # (N_calib,)
selected_features_per_class = cache['selected_features_per_class']  # (n_classes, n_selected)
features_train = cache['features_train']  # (N_train, n_features)
labels_train = cache['labels_train']  # (N_train,)
n_selected = cache['selected_n_shap_features'].item()

n_samples, n_features, n_classes = shap_values.shape

print(f"SHAP Cache Summary:")
print(f"  Calibration samples: {n_samples}")
print(f"  Features: {n_features}")
print(f"  Classes: {n_classes}")
print(f"  Selected features per class: {n_selected}")
print(f"  Training samples: {len(features_train)}")
print(f"  Dataset: {DATASET_FLAG} ({MODEL_BACKBONE}, fold {FOLD_IDX})")
print(f"\nClass distribution (calib):")
unique, counts = np.unique(labels, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Class {cls}: {cnt} samples")


## 2. Feature Importance: Mean Absolute SHAP Values

In [ ]:
# Compute mean absolute SHAP values per feature per class
# Shape: (n_features, n_classes)
mean_abs_shap = np.abs(shap_values).mean(axis=0)

# Also compute for selected features only
selected_feature_importance = {}
for cls in range(n_classes):
    selected_idx_raw = selected_features_per_class[cls]
    # Convert from 'Feature_N' format to integer indices
    if isinstance(selected_idx_raw[0], str):
        selected_idx = np.array([int(s.split('_')[1]) for s in selected_idx_raw], dtype=int)
    else:
        selected_idx = np.array(selected_idx_raw, dtype=int)
    importance = mean_abs_shap[selected_idx, cls]
    selected_feature_importance[cls] = importance

print(f"Mean Absolute SHAP Values (ALL features):")
print(f"  Global mean per feature (avg across classes): {mean_abs_shap.mean(axis=1)[:10]}...")
print(f"\nTop-5 Most Important Features (overall):")
global_importance = mean_abs_shap.mean(axis=1)
top_features = np.argsort(global_importance)[-5:][::-1]
for rank, feat_idx in enumerate(top_features, 1):
    print(f"  {rank}. Feature {feat_idx}: {global_importance[feat_idx]:.4f}")

## 3. Barplot: Mean SHAP Feature Importance

In [ ]:
# Plot: Mean SHAP Feature Importance - One subplot per class, ALL features
# Stacked vertically with large size
fig, axes = plt.subplots(n_classes, 1, figsize=(18, 4*n_classes))

# Global min/max for consistent scaling across all subplots
global_importance = mean_abs_shap.mean(axis=1)
global_max = global_importance.max()

for cls in range(n_classes):
    ax = axes[cls]
    
    # Get importance for this class
    class_importance = mean_abs_shap[:, cls]
    
    # Create bar plot with all features in native order
    colors = ['steelblue' if imp > 0 else 'lightgray' for imp in class_importance]
    bars = ax.bar(range(n_features), class_importance, color=colors, width=0.8)
    
    ax.set_xlabel('Feature Index', fontsize=11)
    ax.set_ylabel('Mean |SHAP| Value', fontsize=11)
    ax.set_title(f'Class {cls}', fontsize=13, fontweight='bold')
    ax.set_ylim(0, global_max * 1.1)
    
    # Add grid for readability
    ax.grid(axis='y', alpha=0.3)
    ax.set_axisbelow(True)
    
    # Reduce x-axis ticks for clarity
    step = max(1, n_features // 15)
    ax.set_xticks(range(0, n_features, step))
    
    # Add statistics box
    top_idx = np.argmax(class_importance)
    top_val = class_importance[top_idx]
    mean_val = class_importance.mean()
    std_val = class_importance.std()
    
    textstr = f'Top: F{top_idx} ({top_val:.3f})\nMean: {mean_val:.3f}\nStd: {std_val:.3f}'
    ax.text(0.98, 0.97, textstr, transform=ax.transAxes,
            fontsize=10, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

fig.suptitle(f'Mean Absolute SHAP Values per Feature - One subplot per class\n({n_features} features, {n_classes} classes)', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('/workspace/knn_shap_all_features_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ All-features barplot saved to /workspace/knn_shap_all_features_per_class.png")

## 4. Selected Features Analysis

In [ ]:
# Analyze selected features per class
print("\nSelected Features per Class:")
print("="*80)

for cls in range(n_classes):
    selected_idx_raw = selected_features_per_class[cls]
    # Convert from 'Feature_N' format to integer indices
    if isinstance(selected_idx_raw[0], str):
        selected_idx = np.array([int(s.split('_')[1]) for s in selected_idx_raw], dtype=int)
    else:
        selected_idx = np.array(selected_idx_raw, dtype=int)
    
    importance = mean_abs_shap[selected_idx, cls]
    
    # Sort selected features by importance
    sort_order = np.argsort(importance)[::-1]
    
    print(f"\nClass {cls}: {len(selected_idx)} selected features")
    print(f"  Top-5 selected features by importance:")
    for rank, idx_pos in enumerate(sort_order[:5], 1):
        feat_idx = selected_idx[idx_pos]
        shap_imp = importance[idx_pos]
        print(f"    {rank}. Feature {feat_idx}: |SHAP| = {shap_imp:.4f}")

# Analyze feature overlap between classes
print(f"\n\nFeature Selection Overlap Analysis:")
print("="*80)

# Create a matrix showing which classes select which features
feature_selection_matrix = np.zeros((n_classes, n_features), dtype=bool)
for cls in range(n_classes):
    selected_idx_raw = selected_features_per_class[cls]
    if isinstance(selected_idx_raw[0], str):
        selected_idx = np.array([int(s.split('_')[1]) for s in selected_idx_raw], dtype=int)
    else:
        selected_idx = np.array(selected_idx_raw, dtype=int)
    feature_selection_matrix[cls, selected_idx] = True

# Count how many classes select each feature
num_classes_selecting = feature_selection_matrix.sum(axis=0)

print(f"\nFeature selection coverage:")
print(f"  Features selected by all classes: {(num_classes_selecting == n_classes).sum()}")
print(f"  Features selected by >50% classes: {(num_classes_selecting > n_classes/2).sum()}")
print(f"  Features selected by single class only: {(num_classes_selecting == 1).sum()}")

# Pairwise overlap between classes
print(f"\nPairwise feature overlap between classes:")
overlap_matrix = np.zeros((n_classes, n_classes))
for c1 in range(n_classes):
    for c2 in range(n_classes):
        idx1_raw = selected_features_per_class[c1]
        idx2_raw = selected_features_per_class[c2]
        
        if isinstance(idx1_raw[0], str):
            set1 = set([int(s.split('_')[1]) for s in idx1_raw])
        else:
            set1 = set(idx1_raw)
            
        if isinstance(idx2_raw[0], str):
            set2 = set([int(s.split('_')[1]) for s in idx2_raw])
        else:
            set2 = set(idx2_raw)
            
        overlap = len(set1 & set2) / max(len(set1), len(set2))  # Jaccard-like
        overlap_matrix[c1, c2] = overlap

print("  Pairwise overlap (features selected in both):")
for c1 in range(min(n_classes, 7)):
    for c2 in range(c1+1, min(n_classes, 7)):
        overlap = overlap_matrix[c1, c2]
        idx1_raw = selected_features_per_class[c1]
        idx2_raw = selected_features_per_class[c2]
        
        if isinstance(idx1_raw[0], str):
            set1 = set([int(s.split('_')[1]) for s in idx1_raw])
        else:
            set1 = set(idx1_raw)
            
        if isinstance(idx2_raw[0], str):
            set2 = set([int(s.split('_')[1]) for s in idx2_raw])
        else:
            set2 = set(idx2_raw)
            
        common = len(set1 & set2)
        print(f"    Class {c1} ↔ Class {c2}: {common}/{n_selected} = {overlap:.2%}")


## 5. Heatmap: Feature Selection by Class

In [ ]:
# Visualize feature selection patterns
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Feature overlap heatmap
ax = axes[0]
im = ax.imshow(overlap_matrix, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(n_classes))
ax.set_yticks(range(n_classes))
ax.set_xticklabels([f'Class {i}' for i in range(n_classes)])
ax.set_yticklabels([f'Class {i}' for i in range(n_classes)])
ax.set_title('Feature Selection Overlap Between Classes', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, label='Jaccard Similarity')

# Add text annotations
for i in range(n_classes):
    for j in range(n_classes):
        text = ax.text(j, i, f'{overlap_matrix[i, j]:.2f}',
                       ha="center", va="center", color="black" if overlap_matrix[i, j] < 0.5 else "white",
                       fontsize=9)

# Plot 2: How many classes select each feature (for top features)
ax = axes[1]
top_n_features_plot = 30
top_indices_plot = np.argsort(global_importance)[-top_n_features_plot:][::-1]
num_classes_selecting_top = num_classes_selecting[top_indices_plot]

colors = ['red' if x == 1 else 'orange' if x < n_classes/2 else 'green' for x in num_classes_selecting_top]
ax.barh(range(top_n_features_plot), num_classes_selecting_top, color=colors)
ax.set_yticks(range(top_n_features_plot))
ax.set_yticklabels([f'Feature {i}' for i in top_indices_plot], fontsize=9)
ax.set_xlabel('Number of Classes Selecting Feature', fontsize=11)
ax.set_title(f'Feature Coverage Across Classes (Top-{top_n_features_plot} Features)', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.set_xticks(range(n_classes+1))
for i, val in enumerate(num_classes_selecting_top):
    ax.text(val, i, f' {int(val)}/{n_classes}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('/workspace/knn_shap_feature_selection_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Feature selection analysis plots saved to /workspace/knn_shap_feature_selection_analysis.png")

## 6. Diagnostic: Are Selected Features Actually Useful for KNN?

In [ ]:
# Check if selected features have high variance/discriminative power
print("\nDiagnostic: Feature Discriminative Power")
print("="*80)

# For each class, check variance of selected features within class vs between classes
for cls in range(n_classes):
    selected_idx_raw = selected_features_per_class[cls]
    # Convert from 'Feature_N' format to integer indices
    if isinstance(selected_idx_raw[0], str):
        selected_idx = np.array([int(s.split('_')[1]) for s in selected_idx_raw], dtype=int)
    else:
        selected_idx = np.array(selected_idx_raw, dtype=int)
    
    class_mask = labels == cls
    
    # Features for this class
    class_features = features[class_mask][:, selected_idx]
    other_features = features[~class_mask][:, selected_idx]
    
    # Calculate separability metric (between-class variance / within-class variance)
    within_class_var = class_features.var(axis=0).mean()
    between_class_var = np.abs(class_features.mean(axis=0) - other_features.mean(axis=0)).mean()
    
    discriminative_ratio = between_class_var / (within_class_var + 1e-10)
    
    print(f"\nClass {cls}:")
    print(f"  Within-class variance (mean): {within_class_var:.4f}")
    print(f"  Between-class separation: {between_class_var:.4f}")
    print(f"  Discriminative ratio: {discriminative_ratio:.4f}")
    print(f"  Samples in class: {class_mask.sum()} / {len(labels)}")


## 7. Comparison: All Features vs Selected Features

In [ ]:
# Compare KNN performance using all features vs selected features
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

print("\nKNN Distance Analysis: All Features vs Selected Features")
print("="*80)

# Standardize features
scaler_all = StandardScaler()
features_train_scaled = scaler_all.fit_transform(features_train)
features_scaled = scaler_all.transform(features)

# For each class, measure average KNN distance using selected features
print("\nPer-Class KNN Distance Comparison:")
print(f"{'Class':<6} {'Samples':<8} {'All Features':<20} {'Selected Features':<20} {'Ratio':<8}")
print("-" * 70)

for cls in range(n_classes):
    selected_idx_raw = selected_features_per_class[cls]
    # Convert from 'Feature_N' format to integer indices
    if isinstance(selected_idx_raw[0], str):
        selected_idx = np.array([int(s.split('_')[1]) for s in selected_idx_raw], dtype=int)
    else:
        selected_idx = np.array(selected_idx_raw, dtype=int)
    
    class_mask = labels == cls
    n_samples = class_mask.sum()
    
    # Skip if too few samples
    if n_samples < 2:
        continue
    
    # ===== ALL FEATURES KNN =====
    X_train_all = features_train_scaled
    X_calib_all = features_scaled[class_mask]
    
    knn_all = NearestNeighbors(n_neighbors=min(5, len(X_train_all)-1), metric='euclidean')
    knn_all.fit(X_train_all)
    distances_all, _ = knn_all.kneighbors(X_calib_all)
    avg_dist_all = distances_all.mean()
    
    # ===== SELECTED FEATURES KNN =====
    X_train_selected = features_train_scaled[:, selected_idx]
    X_calib_selected = features_scaled[class_mask][:, selected_idx]
    
    knn_selected = NearestNeighbors(n_neighbors=min(5, len(X_train_selected)-1), metric='euclidean')
    knn_selected.fit(X_train_selected)
    distances_selected, _ = knn_selected.kneighbors(X_calib_selected)
    avg_dist_selected = distances_selected.mean()
    
    # Ratio: selected vs all (higher = selected features give larger distances = more uncertainty)
    ratio = avg_dist_selected / (avg_dist_all + 1e-10)
    
    print(f"{cls:<6} {n_samples:<8} {avg_dist_all:<20.4f} {avg_dist_selected:<20.4f} {ratio:<8.2f}x")

print("\n💡 Interpretation:")
print("  Ratio > 1.0: Selected features → larger KNN distances (higher uncertainty)")
print("  Ratio ≈ 1.0: Selected features ≈ all features (similar performance)")
print("  Ratio < 1.0: Selected features → smaller KNN distances (lower uncertainty)")
print("\nIf selected features consistently give larger distances, they may be")
print("TOO RESTRICTIVE → losing signal → KNN can't find good neighbors")

## 8. Summary & Hypothesis

In [ ]:
## 12. CRITICAL: Are Selected Features Actually Discriminative for Each Class?

print("\n" + "="*80)
print("CRITICAL DIAGNOSTIC: Discriminative Power of Selected Features")
print("="*80)
print("(Question: Do SHAP-selected features actually separate classes well?)")

from sklearn.preprocessing import StandardScaler

# For each class, check if selected features separate it from others
for cls in range(n_classes):
    selected_idx_raw = selected_features_per_class[cls]
    if isinstance(selected_idx_raw[0], str):
        selected_idx = np.array([int(s.split('_')[1]) for s in selected_idx_raw], dtype=int)
    else:
        selected_idx = np.array(selected_idx_raw, dtype=int)
    
    class_mask = labels == cls
    n_class_samples = class_mask.sum()
    
    # Features for this class
    X_class = features[class_mask][:, selected_idx]
    X_other = features[~class_mask][:, selected_idx]
    
    # Standardize
    scaler = StandardScaler()
    X_class_scaled = scaler.fit_transform(X_class)
    X_other_scaled = scaler.transform(X_other)
    
    # Discriminative metrics
    within_class_var = X_class_scaled.var()  # Pooled variance within class
    between_class_sep = np.linalg.norm(X_class_scaled.mean(axis=0) - X_other_scaled.mean(axis=0))
    
    # Fisher's discriminant ratio (higher = better)
    fisher_ratio = between_class_sep**2 / (within_class_var + 1e-10)
    
    print(f"\nClass {cls} (n={n_class_samples}):")
    print(f"  Within-class variance: {within_class_var:.4f}")
    print(f"  Between-class separation: {between_class_sep:.4f}")
    print(f"  Fisher discriminant ratio: {fisher_ratio:.4f}")
    
    if fisher_ratio > 1.0:
        print(f"  ✅ GOOD: Selected features separate this class well")
    elif fisher_ratio > 0.5:
        print(f"  ⚠️  WEAK: Moderate separability")
    else:
        print(f"  ❌ POOR: Selected features don't separate this class well!")


In [ ]:
## 11. Per-Class Feature Dissimilarity: Are Selected Features Truly Different?

print("\n" + "="*80)
print("KEY QUESTION: Are selected features DISSIMILAR between classes?")
print("="*80)
print("(If YES → per-class selection is justified. If NO → redundant)")

# For each class, get the selected features
per_class_selections = {}
for cls in range(n_classes):
    selected_idx_raw = selected_features_per_class[cls]
    if isinstance(selected_idx_raw[0], str):
        selected_idx = [int(s.split('_')[1]) for s in selected_idx_raw]
    else:
        selected_idx = list(selected_idx_raw)
    per_class_selections[cls] = set(selected_idx)

print(f"\nPer-class feature selections ({n_selected} features selected per class):")
for cls in range(n_classes):
    selected = per_class_selections[cls]
    print(f"  Class {cls}: {len(selected)} features selected")

# Compute pairwise dissimilarity (using Jaccard distance = 1 - Jaccard similarity)
import itertools
pairwise_dissim = {}
pairwise_overlap = {}
for c1, c2 in itertools.combinations(range(n_classes), 2):
    s1 = per_class_selections[c1]
    s2 = per_class_selections[c2]
    intersection = len(s1 & s2)
    union = len(s1 | s2)
    jaccard_sim = intersection / union if union > 0 else 0
    jaccard_dist = 1 - jaccard_sim
    pairwise_dissim[(c1, c2)] = jaccard_dist
    pairwise_overlap[(c1, c2)] = intersection

print(f"\nPairwise feature overlap (all class pairs):")
dissim_values = []
overlap_values = []
n_selected_denom = max(1, n_selected)
for (c1, c2), overlap in sorted(pairwise_overlap.items()):
    dissim = pairwise_dissim[(c1, c2)]
    dissim_values.append(dissim)
    overlap_values.append(overlap)
    print(f"  Class {c1} ↔ Class {c2}: {overlap} shared / {n_selected_denom} each = {overlap/n_selected_denom:.1%} overlap, {dissim:.2%} dissimilarity")

avg_overlap = np.mean(overlap_values)
avg_dissim = np.mean(dissim_values)

print(f"\n📊 Summary Statistics:")
print(f"  Average pairwise overlap: {avg_overlap:.1f} features ({avg_overlap/n_selected_denom:.1%})")
print(f"  Average pairwise dissimilarity: {avg_dissim:.1%}")
print(f"  Min/Max dissimilarity: {min(dissim_values):.1%} / {max(dissim_values):.1%}")

# How many features are UNIQUE to each class?
print(f"\nClass-Specific Features (selected only by that class):")
for cls in range(n_classes):
    cls_features = per_class_selections[cls]
    # Features NOT selected by any other class
    unique_to_cls = cls_features.copy()
    for other_cls in range(n_classes):
        if other_cls != cls:
            unique_to_cls -= per_class_selections[other_cls]
    print(f"  Class {cls}: {len(unique_to_cls)} unique features")

print(f"\n💡 Interpretation:")
if avg_dissim > 0.6:
    print(f"  ✅ HIGH dissimilarity ({avg_dissim:.1%}) → Per-class selection is JUSTIFIED!")
    print(f"     Different classes rely on fundamentally different features.")
elif avg_dissim > 0.4:
    print(f"  ⚠️  MODERATE dissimilarity ({avg_dissim:.1%}) → Partially justified.")
    print(f"     Some features are shared, some are class-specific.")
else:
    print(f"  ❌ LOW dissimilarity ({avg_dissim:.1%}) → Per-class selection is REDUNDANT.")
    print(f"     Most important features are the same across classes.")


In [ ]:
## Load fold model + real test set, compute RAW (pre-standardization) KNN-SHAP distances
import sys
sys.path.insert(0, '/workspace')

import random
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

from Benchmarks.medMNIST.utils import train_models_load_datasets as tr
from Benchmarks.medMNIST.utils.data_preprocessing_classification_evaluation import dataset_utils
from ToolBox.methods.latent import (
    get_layer_from_model,
    extract_latent_space_and_compute_shap_importance,
)

# Match the benchmark script's determinism settings exactly (seed=42, deterministic cudnn),
# otherwise convolution algorithm nondeterminism flips a handful of borderline predictions.
random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Uses DATASET_FLAG/MODEL_BACKBONE/FOLD_IDX from the configuration cell (matches the loaded SHAP cache)
models = tr.load_models(DATASET_FLAG, device=device, size=IMAGE_SIZE, model_backbone=MODEL_BACKBONE, setup='')
model_fold = models[FOLD_IDX]
model_fold.eval()

transform, _ = dataset_utils.get_transforms(COLOR, IMAGE_SIZE)
[_, _, test_dataset], [_, _, test_loader], info = tr.load_datasets(
    DATASET_FLAG, COLOR, IMAGE_SIZE, transform, batch_size=256, test_subset='all'
)

layer = get_layer_from_model(model_fold, 'avgpool')
features_test, labels_test, _, predicted_raw = extract_latent_space_and_compute_shap_importance(
    model_fold, test_loader, device, layer, importance=False
)
labels_test = np.array(labels_test, dtype=int)
predicted_raw = np.array(predicted_raw)
# predicted_raw is softmax probabilities per class (multi-class) or a single sigmoid
# probability (binary). Rounding BEFORE argmax is only valid for binary; for multi-class
# it destroys the ranking whenever the top class has <50% probability (all-zero tie ->
# argmax silently picks class 0). Use plain argmax for multi-class, round only for binary.
if predicted_raw.ndim > 1:
    predicted_classes = np.argmax(predicted_raw, axis=1)
else:
    predicted_classes = np.round(predicted_raw).astype(int)

print(f"Test samples: {len(labels_test)}")
print(f"Overall fold-{FOLD_IDX} accuracy: {(predicted_classes == labels_test).mean():.4f}")

# Sanity check against the pre-cached benchmark predictions for this fold (same model/order)
_cache_test = np.load(TEST_CACHE_FILE, allow_pickle=True)
_match = np.array_equal(labels_test, _cache_test['y_true'])
print(f"labels_test matches cached y_true order: {_match}")
if _match:
    _agree = (predicted_classes == _cache_test['per_fold_predictions'][FOLD_IDX]).mean()
    print(f"Predicted-class agreement with cached fold-{FOLD_IDX} predictions: {_agree:.4f}")


In [ ]:
## Fit per-class KNN (train fold) and compute RAW (un-standardized) test distances, split correct/incorrect
# Use the CACHED benchmark predictions/labels (not the freshly re-run ones) so results are
# directly comparable to the historic baseline, sidestepping the ~0.3% prediction
# drift from environment/package differences between now and when the cache was generated.
_cache_test = np.load(TEST_CACHE_FILE, allow_pickle=True)
labels_test = _cache_test['y_true']
predicted_classes = _cache_test['per_fold_predictions'][FOLD_IDX]
assert len(labels_test) == len(features_test), "cached test set size must match extracted features"
print(f"Using CACHED labels/predictions: accuracy={np.mean(predicted_classes == labels_test):.4f} "
      f"(matches historic fold-{FOLD_IDX} accuracy)")

train_df = pd.DataFrame(features_train, columns=[f"Feature_{i}" for i in range(features_train.shape[1])])
test_df = pd.DataFrame(features_test.numpy(), columns=[f"Feature_{i}" for i in range(features_test.shape[1])])

n_classes_bar = len(selected_features_per_class)
raw_correct_by_class = {c: [] for c in range(n_classes_bar)}
raw_incorrect_by_class = {c: [] for c in range(n_classes_bar)}

for class_idx in range(n_classes_bar):
    sel_feats_raw = selected_features_per_class[class_idx]
    sel_feats = [f"Feature_{int(s.split('_')[1])}" if isinstance(s, str) else f"Feature_{int(s)}" for s in sel_feats_raw]

    train_mask = (labels_train == class_idx)
    if train_mask.sum() == 0:
        continue
    train_selected = train_df[train_mask][sel_feats].values

    scaler = StandardScaler()
    train_std = scaler.fit_transform(train_selected)
    pca = PCA(n_components=0.9)
    train_pca = pca.fit_transform(train_std)
    knn = NearestNeighbors(n_neighbors=min(5, len(train_pca)))
    knn.fit(train_pca)

    pred_mask = (predicted_classes == class_idx)
    if pred_mask.sum() == 0:
        continue
    test_selected = test_df[pred_mask][sel_feats].values
    test_std = scaler.transform(test_selected)
    test_pca = pca.transform(test_std)
    distances, _ = knn.kneighbors(test_pca)
    avg_distances = distances.mean(axis=1)  # RAW, no normalization

    true_labels_here = labels_test[pred_mask]
    raw_correct_by_class[class_idx] = avg_distances[true_labels_here == class_idx]
    raw_incorrect_by_class[class_idx] = avg_distances[true_labels_here != class_idx]

    print(f"Class {class_idx}: n_pred={pred_mask.sum()} correct={len(raw_correct_by_class[class_idx])} "
          f"incorrect={len(raw_incorrect_by_class[class_idx])}")


In [ ]:
## Barplot + individual sample points: RAW (pre-standardization) correct vs incorrect KNN-SHAP distances, per class
fig, ax = plt.subplots(figsize=(16, 7))

classes_bar = np.arange(n_classes_bar)
width = 0.35
rng_jitter = np.random.default_rng(0)

correct_means = [raw_correct_by_class[c].mean() if len(raw_correct_by_class[c]) else 0 for c in classes_bar]
incorrect_means = [raw_incorrect_by_class[c].mean() if len(raw_incorrect_by_class[c]) else 0 for c in classes_bar]
error_rates = [len(raw_incorrect_by_class[c]) / max(1, len(raw_incorrect_by_class[c]) + len(raw_correct_by_class[c])) for c in classes_bar]

ax.boxplot([raw_correct_by_class[c] for c in classes_bar], positions=classes_bar - width/2, widths=width,
           patch_artist=True, boxprops=dict(facecolor='#2ca02c', alpha=0.5), medianprops=dict(color='black'), zorder=1)
ax.boxplot([raw_incorrect_by_class[c] for c in classes_bar], positions=classes_bar + width/2, widths=width,
           patch_artist=True, boxprops=dict(facecolor='#d62728', alpha=0.5), medianprops=dict(color='black'), zorder=1)

for c in classes_bar:
    pts_c = raw_correct_by_class[c]
    if len(pts_c):
        jitter = rng_jitter.uniform(-width/2 * 0.8, width/2 * 0.8, size=len(pts_c))
        ax.scatter(c - width/2 + jitter, pts_c, s=6, color='#1a6e1a', alpha=0.35, zorder=2, linewidths=0)

    pts_i = raw_incorrect_by_class[c]
    if len(pts_i):
        jitter = rng_jitter.uniform(-width/2 * 0.8, width/2 * 0.8, size=len(pts_i))
        ax.scatter(c + width/2 + jitter, pts_i, s=10, color='#8b0000', alpha=0.6, zorder=2, linewidths=0)

    ax.annotate(f"{error_rates[c]:.1%} error", (c + width/2, ax.get_ylim()[1] * 0.97),
                ha='center', fontsize=8, color='#d62728')

ax.set_xlabel('Predicted class (organ)')
ax.set_ylabel('Raw KNN distance (pre-standardization)')
ax.set_title(f'{DATASET_FLAG} ({MODEL_BACKBONE}) fold-{FOLD_IDX}: RAW KNN-SHAP distances, correct vs incorrect predictions, per class\n'
             '(bars = mean, dots = individual test samples; annotations show each class\'s local misclassification rate)')
ax.set_xticks(classes_bar)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
## Same boxplots, but standardized with the OLD method (test-time mixed correct+incorrect mean/std per class)
old_std_correct_by_class = {}
old_std_incorrect_by_class = {}

for c in classes_bar:
    mixed = np.concatenate([raw_correct_by_class[c], raw_incorrect_by_class[c]])
    old_mean, old_std = mixed.mean(), mixed.std()
    if old_std > 1e-10:
        old_std_correct_by_class[c] = (raw_correct_by_class[c] - old_mean) / old_std
        old_std_incorrect_by_class[c] = (raw_incorrect_by_class[c] - old_mean) / old_std
    else:
        old_std_correct_by_class[c] = np.zeros_like(raw_correct_by_class[c])
        old_std_incorrect_by_class[c] = np.zeros_like(raw_incorrect_by_class[c])

fig, ax = plt.subplots(figsize=(16, 7))

ax.boxplot([old_std_correct_by_class[c] for c in classes_bar], positions=classes_bar - width/2, widths=width,
           patch_artist=True, boxprops=dict(facecolor='#2ca02c', alpha=0.5), medianprops=dict(color='black'), zorder=1)
ax.boxplot([old_std_incorrect_by_class[c] for c in classes_bar], positions=classes_bar + width/2, widths=width,
           patch_artist=True, boxprops=dict(facecolor='#d62728', alpha=0.5), medianprops=dict(color='black'), zorder=1)

for c in classes_bar:
    pts_c = old_std_correct_by_class[c]
    if len(pts_c):
        jitter = rng_jitter.uniform(-width/2 * 0.8, width/2 * 0.8, size=len(pts_c))
        ax.scatter(c - width/2 + jitter, pts_c, s=6, color='#1a6e1a', alpha=0.35, zorder=2, linewidths=0)

    pts_i = old_std_incorrect_by_class[c]
    if len(pts_i):
        jitter = rng_jitter.uniform(-width/2 * 0.8, width/2 * 0.8, size=len(pts_i))
        ax.scatter(c + width/2 + jitter, pts_i, s=10, color='#8b0000', alpha=0.6, zorder=2, linewidths=0)

    ax.annotate(f"{error_rates[c]:.1%} error", (c + width/2, ax.get_ylim()[1] * 0.97),
                ha='center', fontsize=8, color='#d62728')

ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('Predicted class (organ)')
ax.set_ylabel('Z-score (OLD: mixed test-time correct+incorrect mean/std per class)')
ax.set_title(f'{DATASET_FLAG} ({MODEL_BACKBONE}) fold-{FOLD_IDX}: OLD standardization, correct vs incorrect predictions, per class\n'
             '(bars = mean, dots = individual test samples; annotations show each class\'s local misclassification rate)')
ax.set_xticks(classes_bar)
plt.tight_layout()
plt.show()


In [ ]:
## Same boxplots, standardized with MIDPOINT-of-clusters computed from CALIBRATION set (no test leakage)
_cache_calib = np.load(CALIB_CACHE_FILE, allow_pickle=True)
labels_calib_gt = _cache_calib['y_true']
predicted_classes_calib = _cache_calib['per_fold_predictions'][0]  # fold-0, matches loaded SHAP cache
calib_df = pd.DataFrame(features, columns=[f"Feature_{i}" for i in range(features.shape[1])])

midpoint_std_correct_by_class = {}
midpoint_std_incorrect_by_class = {}

for c in classes_bar:
    sel_feats_raw = selected_features_per_class[c]
    sel_feats = [f"Feature_{int(s.split('_')[1])}" if isinstance(s, str) else f"Feature_{int(s)}" for s in sel_feats_raw]

    train_mask = (labels_train == c)
    train_selected = train_df[train_mask][sel_feats].values
    scaler = StandardScaler()
    train_std = scaler.fit_transform(train_selected)
    pca = PCA(n_components=0.9)
    train_pca = pca.fit_transform(train_std)
    knn = NearestNeighbors(n_neighbors=min(5, len(train_pca)))
    knn.fit(train_pca)

    correct_c = raw_correct_by_class[c]
    incorrect_c = raw_incorrect_by_class[c]

    # Calibration samples PREDICTED as class c -> distances -> split by CALIB true label
    calib_pred_mask = (predicted_classes_calib == c)
    if calib_pred_mask.sum() == 0:
        # No calibration signal at all for this predicted class: nothing to normalize with.
        midpoint_std_correct_by_class[c] = np.zeros_like(correct_c)
        midpoint_std_incorrect_by_class[c] = np.zeros_like(incorrect_c)
        continue

    calib_selected = calib_df[calib_pred_mask][sel_feats].values
    calib_pca = pca.transform(scaler.transform(calib_selected))
    calib_distances, _ = knn.kneighbors(calib_pca)
    calib_avg_distances = calib_distances.mean(axis=1)
    calib_true_here = labels_calib_gt[calib_pred_mask]
    calib_correct = calib_avg_distances[calib_true_here == c]
    calib_incorrect = calib_avg_distances[calib_true_here != c]

    if len(calib_correct) > 0 and len(calib_incorrect) > 0:
        mid_mean = (calib_correct.mean() + calib_incorrect.mean()) / 2  # true midpoint of both clusters
        mixed_calib = np.concatenate([calib_correct, calib_incorrect])
    else:
        # Calib has only one outcome for this class (e.g. never wrong on calib): fall back to
        # the mean/std of all available calib distances instead of zeroing the class out.
        mid_mean = calib_avg_distances.mean()
        mixed_calib = calib_avg_distances
    mid_std = mixed_calib.std()

    if mid_std > 1e-10:
        midpoint_std_correct_by_class[c] = (correct_c - mid_mean) / mid_std
        midpoint_std_incorrect_by_class[c] = (incorrect_c - mid_mean) / mid_std
    else:
        midpoint_std_correct_by_class[c] = np.zeros_like(correct_c)
        midpoint_std_incorrect_by_class[c] = np.zeros_like(incorrect_c)

fig, ax = plt.subplots(figsize=(16, 7))

ax.boxplot([midpoint_std_correct_by_class[c] for c in classes_bar], positions=classes_bar - width/2, widths=width,
           patch_artist=True, boxprops=dict(facecolor='#2ca02c', alpha=0.5), medianprops=dict(color='black'), zorder=1)
ax.boxplot([midpoint_std_incorrect_by_class[c] for c in classes_bar], positions=classes_bar + width/2, widths=width,
           patch_artist=True, boxprops=dict(facecolor='#d62728', alpha=0.5), medianprops=dict(color='black'), zorder=1)

for c in classes_bar:
    pts_c = midpoint_std_correct_by_class[c]
    if len(pts_c):
        jitter = rng_jitter.uniform(-width/2 * 0.8, width/2 * 0.8, size=len(pts_c))
        ax.scatter(c - width/2 + jitter, pts_c, s=6, color='#1a6e1a', alpha=0.35, zorder=2, linewidths=0)

    pts_i = midpoint_std_incorrect_by_class[c]
    if len(pts_i):
        jitter = rng_jitter.uniform(-width/2 * 0.8, width/2 * 0.8, size=len(pts_i))
        ax.scatter(c + width/2 + jitter, pts_i, s=10, color='#8b0000', alpha=0.6, zorder=2, linewidths=0)

    ax.annotate(f"{error_rates[c]:.1%} error", (c + width/2, ax.get_ylim()[1] * 0.97),
                ha='center', fontsize=8, color='#d62728')

ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('Predicted class (organ)')
ax.set_ylabel('Z-score (MIDPOINT: calib-derived mean of correct/incorrect cluster means)')
ax.set_title('OrganaMNIST fold-0: CALIB-derived MIDPOINT standardization, correct vs incorrect test predictions, per class\n'
             '(bars = mean, dots = individual test samples; annotations show each class\'s local misclassification rate)')
ax.set_xticks(classes_bar)
plt.tight_layout()
plt.show()


In [ ]:
## Pooled AUROC on REAL data: RAW (no normalization) vs OLD vs MIDPOINT vs CALIBRATION-based reference
from sklearn.metrics import roc_auc_score

calib_df = pd.DataFrame(features, columns=[f"Feature_{i}" for i in range(features.shape[1])])

calib_std_correct_by_class = {}
calib_std_incorrect_by_class = {}

for c in classes_bar:
    sel_feats_raw = selected_features_per_class[c]
    sel_feats = [f"Feature_{int(s.split('_')[1])}" if isinstance(s, str) else f"Feature_{int(s)}" for s in sel_feats_raw]

    train_mask = (labels_train == c)
    train_selected = train_df[train_mask][sel_feats].values
    scaler = StandardScaler()
    train_std = scaler.fit_transform(train_selected)
    pca = PCA(n_components=0.9)
    train_pca = pca.fit_transform(train_std)
    knn = NearestNeighbors(n_neighbors=min(5, len(train_pca)))
    knn.fit(train_pca)

    # Reference stats from calibration set, true label == c only (matches the implemented fix)
    calib_mask = (labels == c)
    calib_selected = calib_df[calib_mask][sel_feats].values
    calib_pca = pca.transform(scaler.transform(calib_selected))
    ref_distances, _ = knn.kneighbors(calib_pca)
    ref_avg = ref_distances.mean(axis=1)
    ref_mean, ref_std = ref_avg.mean(), ref_avg.std()

    if ref_std > 1e-10:
        calib_std_correct_by_class[c] = (raw_correct_by_class[c] - ref_mean) / ref_std
        calib_std_incorrect_by_class[c] = (raw_incorrect_by_class[c] - ref_mean) / ref_std
    else:
        calib_std_correct_by_class[c] = np.zeros_like(raw_correct_by_class[c])
        calib_std_incorrect_by_class[c] = np.zeros_like(raw_incorrect_by_class[c])

def pooled_auroc(correct_by_class, incorrect_by_class):
    scores, labels_bin = [], []
    for c in classes_bar:
        scores.append(correct_by_class[c]); labels_bin.append(np.zeros(len(correct_by_class[c])))
        scores.append(incorrect_by_class[c]); labels_bin.append(np.ones(len(incorrect_by_class[c])))
    scores = np.concatenate(scores)
    labels_bin = np.concatenate(labels_bin)
    return roc_auc_score(labels_bin, scores)

auroc_raw = pooled_auroc(raw_correct_by_class, raw_incorrect_by_class)
auroc_old = pooled_auroc(old_std_correct_by_class, old_std_incorrect_by_class)
auroc_midpoint = pooled_auroc(midpoint_std_correct_by_class, midpoint_std_incorrect_by_class)
auroc_calib = pooled_auroc(calib_std_correct_by_class, calib_std_incorrect_by_class)

print("Pooled AUROC on REAL OrganaMNIST fold-0 test set (11 classes pooled together):")
print(f"  RAW       (no normalization at all, pooled across classes)  : {auroc_raw:.4f}")
print(f"  OLD       (mixed test-time correct+incorrect mean/std)      : {auroc_old:.4f}")
print(f"  MIDPOINT  (avg of correct-mean & incorrect-mean, calib-only): {auroc_midpoint:.4f}")
print(f"  CALIB-REF (implemented fix: calib true-label-only ref)      : {auroc_calib:.4f}")
